In [1]:
import datetime
import os
import uuid

from dotenv import load_dotenv
import xarray as xr
from xsdba.adjustment import QuantileDeltaMapping, TrainAdjust
from xsdba.base import Grouper

In [2]:
#load_dotenv()

FORECAST_URI = "/home/emily_zuetell/projects/poreallas/data/parsed/08_ecmwf_parsed.zarr"
GMFD_URI = "/home/emily_zuetell/projects/poreallas/data/parsed/gmfd_parsed.zarr"
OUT_ZARR = "/home/emily_zuetell/projects/poreallas/data/forecast_adj.zarr"
HISTREF_START_YEAR = 1981
HISTREF_STOP_YEAR = 1997
SIM_START_YEAR = 2007
SIM_STOP_YEAR = 2027
QDM_N_QUANTILES = 100
FORECAST_LENGTH = 215  # ECMWF S51 is 215 days.
UID = str(uuid.uuid4())
START_TIME = datetime.datetime.now(datetime.UTC).isoformat()



In [6]:
def adjust_month(
    *,
    ref: xr.DataArray,
    hist: xr.DataArray,
    sim: xr.DataArray,
    target_month: int,
    nquantiles: int,
) -> tuple[TrainAdjust, xr.DataArray]:
    """
    Train and apply QDM for a particular `time.month`
    """
    ref = ref.where(ref["time.month"] == target_month, drop=True)
    hist = hist.where(hist["time.month"] == target_month, drop=True)
    sim = sim.where(sim["time.month"] == target_month, drop=True)

    # Check for rollover months, reduce years in ref
    ref = ref.where(ref["time"].isin(hist["time"]), drop=True)

    qdm = QuantileDeltaMapping.train(
        ref,
        hist,
        nquantiles=nquantiles,
        kind="+",
        group=Grouper("time", add_dims=["number"]),
    )
    adj = qdm.adjust(sim)

    return qdm, adj


def adjust_months(
    *,
    ref: xr.DataArray,
    hist: xr.DataArray,
    sim: xr.DataArray,
    nquantiles: int,
) -> xr.DataArray:
    """
    Train and apply quantile delta mapping (QDM) for all `time.month` in a simulation.

    We need a custom algorithm for this because our forecast ensembles run for <
    365 days yet this QDM implementation does not allow us to group by "time.month"
    when it does not have all 12 months. So we train and apply QDM to each of the
    months in the simulation dataset and then concatenate them back together along
    the time dimension. The concatenated data is then sorted by the time dimension
    to return the data to chronological order.

    Parameters
    ----------
    ref :
        Reference dataset to compare against a historical simulation to train a QDM.
    hist :
        Historical simulation dataset to be compared against ref when training the QDM.
    sim :
        Simulation, or forecast ensemble to be adjusted by the trained QDM.
    nquantiles :
        Number of quantiles to use in the quantile mapping.

    Returns
    -------
    combined :
        Simulated, bias-adjusted by a QDM trained on a historical and reference dataset.
    """
    adjusted = []
    for m in set(sim["time.month"].data):
        _, adj = adjust_month(
            ref=ref,
            hist=hist,
            sim=sim,
            target_month=m,
            nquantiles=nquantiles,
        )
        adjusted.append(adj)

    combined = xr.concat(adjusted, dim="time").sortby("time")
    return combined

In [3]:
gmfd = xr.open_zarr(GMFD_URI)
# Fill extreme values
gmfd = gmfd.sortby("latitude").chunk({"latitude": -1, "longitude": 30, "time": -1})
gmfd = gmfd.where(gmfd["tas"] < 1000).interpolate_na(dim="latitude", method="linear").compute()

forecast = xr.open_zarr(FORECAST_URI)

In [7]:
# Outline the datasets we need for the adjustment, grabbing the windows in time needed.
ref = gmfd.sel(time=slice(str(HISTREF_START_YEAR), str(HISTREF_STOP_YEAR)))
hist = forecast.sel(time=slice(str(HISTREF_START_YEAR), str(HISTREF_STOP_YEAR)))
sim = forecast.sel(time=slice(str(SIM_START_YEAR), str(SIM_STOP_YEAR)))

In [8]:
sim_adj = adjust_months(
    ref=ref["tas"],
    hist=hist["tas"],
    sim=sim["tas"],
    nquantiles=QDM_N_QUANTILES,
)

In [10]:
sim_adj = sim_adj.isel(time=slice(-int(FORECAST_LENGTH), None))

In [5]:
# # Subset reference to only daysofyear that are in our forecast ensemble. The
# forecast ensemble has incomplete years. Ref/hist/sim need to have matching
# ragged ends in their time series for QDM.
ref = ref.where(ref["time.dayofyear"].isin(sim["time.dayofyear"]), drop=True)
hist = hist.where(hist["time.dayofyear"].isin(sim["time.dayofyear"]), drop=True)

# Rechunking because all of "time", or whatever we're grouping QDM on, needs to be in one chunk.
ref = ref.chunk({"time": -1, "latitude": 30, "longitude": "auto"})
hist = hist.chunk({"number": -1, "time": -1, "latitude": 30, "longitude": "auto"})
sim = sim.chunk({"number": -1, "time": -1, "latitude": 30, "longitude": "auto"})


In [10]:
target_month = 8
ref=ref["tas"]
hist=hist["tas"]
sim=sim["tas"]
nquantiles=QDM_N_QUANTILES

ref = ref.where(ref["time.month"] == target_month, drop=True)
hist = hist.where(hist["time.month"] == target_month, drop=True)
sim = sim.where(sim["time.month"] == target_month, drop=True)

# Check for rollover months, (lead time not present in 1981)
ref = ref.where(ref["time"].isin(hist["time"]), drop=True)

qdm = QuantileDeltaMapping.train(
    ref,
    hist,
    nquantiles=nquantiles,
    kind="+",
    group=Grouper("time", add_dims=["number"]),
)



In [ ]:
qdm.ds.to_zarr("08_qdm_gmfd.zarr")

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [31]:
adj = qdm.adjust(sim.where(sim["time.year"] == 2026, drop=True))

In [12]:
sim_adj.name = "tas"
sim_adj = sim_adj.to_dataset()

# Add additional general metadata.
sim_adj.attrs |= {
    "poreallas_created_at": START_TIME,
    "poreallas_uid": UID,
    "poreallas_description": "QDM bias-adjusted forecast ensemble fields",
}
sim_adj["tas"].attrs |= {
    "poreallas_created_at": START_TIME,
    "poreallas_uid": UID,
    "poreallas_description": "QDM bias-adjusted forecast ensemble tas fields",
    "poreallas_adjustment_method": "QDM",
    "poreallas_histref_start_year": HISTREF_START_YEAR,
    "poreallas_histref_stop_year": HISTREF_STOP_YEAR,
    "poreallas_sim_start_year": SIM_START_YEAR,
    "poreallas_sim_stop_year": SIM_STOP_YEAR,
    "poreallas_qdm_nquantiles": QDM_N_QUANTILES,
    "poreallas_ref_uri": GMFD_URI,
    "poreallas_hist_uri": FORECAST_URI,
    "poreallas_sim_uri": FORECAST_URI,
}

sim_adj = sim_adj.chunk("auto")

#sim_adj.to_zarr(f"/home/emily_zuetell/projects/poreallas/data/forecast_adj_test.zarr", consolidated=True)

In [14]:
sim_adj['tas']

<xarray.DataArray 'tas' (latitude: 180, time: 215, longitude: 360, number: 51)> Size: 3GB
dask.array<rechunk-merge, shape=(180, 215, 360, 51), dtype=float32, chunksize=(180, 215, 16, 51), chunktype=numpy.ndarray>
Coordinates:
  * latitude                 (latitude) float64 1kB -89.5 -88.5 ... 88.5 89.5
  * time                     (time) object 2kB 2026-08-02 00:00:00 ... 2027-0...
    forecast_period          (time) timedelta64[ns] 2kB dask.array<chunksize=(215,), meta=np.ndarray>
    forecast_reference_time  (time) datetime64[ns] 2kB dask.array<chunksize=(215,), meta=np.ndarray>
  * longitude                (longitude) float64 3kB 0.5 1.5 2.5 ... 358.5 359.5
  * number                   (number) int64 408B 0 1 2 3 4 5 ... 46 47 48 49 50
Attributes: (12/42)
    GRIB_dataType:                            fc
    GRIB_numberOfPoints:                      64800
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    GRIB_gridType:                            regular_ll
    ...                                       ...
    poreallas_sim_start_year:                 2007
    poreallas_sim_stop_year:                  2027
    poreallas_qdm_nquantiles:                 100
    poreallas_ref_uri:                        /home/emily_zuetell/projects/po...
    poreallas_hist_uri:                       /home/emily_zuetell/projects/po...
    poreallas_sim_uri:                        /home/emily_zuetell/projects/po...

In [35]:
import re
import glob

#Merge monthly files
def month_num(path):
    return int(re.search(r"m(\d+)_forecast_adj\.zarr", path).group(1))

files = sorted(glob.glob("/home/emily_zuetell/projects/poreallas/data/m*_forecast_adj.zarr"), key=month_num)
ds = xr.open_mfdataset(files, engine="zarr", combine="nested", concat_dim="time")
for var in ds.variables:
    ds[var].encoding.pop("chunks", None)
ds = ds.chunk({"time": -1, "latitude": 30, "longitude": 30})
ds.to_zarr("/home/emily_zuetell/projects/poreallas/data/2607_forecast_adj_corrected.zarr", mode="w")

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
